This notebook implements the forward-looking simulation framework developed in Chapter 5 of the thesis.

The objective is to:
1. Construct baseline projections for maritime cereal trade (2025–2030)
2. Apply structured transport and supply-side stress scenarios
3. Evaluate transmission effects on trade volumes and transport costs
4. Perform segmentation analysis of corridor-level exposure

This notebook is fully reproducible and designed for transparent rerunning of scenario parameters.

# 0. Load and set up

In [1]:
import numpy as np
import pandas as pd
import joblib
import json
from datetime import datetime
import plotly.express as px

# 1. Load Panel Data (Gap Year) and Engine

In [2]:
df_fwd_forecast_gapyear = pd.read_parquet(
    "data/df_fwd_forecast_gapyear.parquet"
)
df_bltr_multivar = pd.read_parquet("data/df_bltr_multivar.parquet")
loaded_model_cost_ann = joblib.load("models/ann_model_pipeline.pkl")
trade_ann_pipeline = joblib.load("models/trade_ann_log_model.pkl")

print("Panel shape:", df_fwd_forecast_gapyear.shape)

from scenario_engine import (
    build_future_features,
    apply_combined_system_stress,
    predict_transport_cost,
    predict_trade_volume,
    run_baseline_and_scenario,
    select_affected_corridors
)
from scenario_engine import run_baseline_and_scenario
from features.ann_feature_builder import make_ann_features_for_forecast
from features.model_schema import (
    COST_NUMERIC_FEATURES,
    COST_CATEGORICAL_FEATURES,
    TRADE_NUMERIC_FEATURES,
    TRADE_CATEGORICAL_FEATURES
)
from features.build_features import build_features
def build_ann_cost_wrapper(build_features):
    def wrapper(df):
        return make_ann_features_for_forecast(
            df_raw=df,
            build_features_func=build_features,
            feature_ann_cols=COST_NUMERIC_FEATURES,
            categorical_ann_cols=COST_CATEGORICAL_FEATURES
        )
    return wrapper
ann_feature_wrapper = build_ann_cost_wrapper(build_features)
model_kwargs = dict(
    make_ann_features_for_forecast=ann_feature_wrapper,
    loaded_model_cost_ann=loaded_model_cost_ann,
    feature_ann_cols=COST_NUMERIC_FEATURES,
    categorical_ann_cols=COST_CATEGORICAL_FEATURES,
    numerical_features=TRADE_NUMERIC_FEATURES,
    trade_categorical_features=TRADE_CATEGORICAL_FEATURES,
    trade_ann_pipeline=trade_ann_pipeline
)

Panel shape: (38232, 28)


HELPER FUNCTIONS Dont Change

In [ ]:
# def safe_log(x):
#     return np.log(np.clip(x, 1e-9, None))


# def build_future_baseline(df_panel, base_year, future_years, growth_dict):
#     df_base = df_panel[df_panel["year"] == base_year].copy()

#     future_frames = []

#     for year in future_years:
#         df_year = df_base.copy()
#         df_year["year"] = year

#         t = year - base_year

#         df_year["origin_gdp_usd_current"] *= (1 + growth_dict["gdp_growth"]) ** t
#         df_year["destination_gdp_usd_current"] *= (1 + growth_dict["gdp_growth"]) ** t

#         df_year["origin_population_total"] *= (1 + growth_dict["population_growth"]) ** t
#         df_year["destination_population_total"] *= (1 + growth_dict["population_growth"]) ** t

#         df_year["origin_production_qty"] *= (1 + growth_dict["production_growth"]) ** t
#         df_year["destination_production_qty"] *= (1 + growth_dict["production_growth"]) ** t

#         future_frames.append(df_year)

#     return pd.concat(future_frames, ignore_index=True)


# def apply_stress(df, scenario):
#     df = df.copy()

#     # Transport shocks
#     df["lsbci"] *= (1 + scenario["transport_shocks"]["connectivity_lsbci"])
#     df["fbx_new"] *= (1 + scenario["transport_shocks"]["freight_proxy_fbx"])
#     df["bdi"] *= (1 + scenario["transport_shocks"]["bdi_proxy"])
#     df["cepii_dist"] *= (1 + scenario["transport_shocks"]["distance_proxy"])

#     # Production shocks
#     for country, shock in scenario["country_shocks"]["production_by_origin_country"].items():
#         df.loc[df["origin_isocode"] == country, "origin_production_qty"] *= (1 + shock)

#     return df


# def predict_transport_cost(df):
#     df = df.copy()

#     df["ln_gdp_o"] = safe_log(df["origin_gdp_usd_current"])
#     df["ln_gdp_d"] = safe_log(df["destination_gdp_usd_current"])
#     df["ln_dist"] = safe_log(df["cepii_dist"])
#     df["ln_bdi"] = safe_log(df["bdi"])
#     df["ln_fbx"] = safe_log(df["fbx_new"])

#     X = df[[
#         "ln_gdp_o", "ln_gdp_d", "ln_dist",
#         "lsbci", "ln_bdi", "ln_fbx"
#     ]]

#     ln_cost_pred = loaded_model_cost_ann.predict(X)
#     df["transport_cost_per_ton_pred"] = np.exp(ln_cost_pred)

#     return df


# def predict_trade_volume(df):
#     df = df.copy()

#     df["Log_cost_per_ton_seamodes"] = safe_log(df["transport_cost_per_ton_pred"])
#     df["Log_Origin_GDP"] = safe_log(df["origin_gdp_usd_current"])
#     df["Log_Destination_GDP"] = safe_log(df["destination_gdp_usd_current"])
#     df["Log_origin_production_qty"] = safe_log(df["origin_production_qty"])
#     df["Log_destination_production_qty"] = safe_log(df["destination_production_qty"])

#     df["country_pair_id"] = (
#         df["origin_isocode"] + "_" + df["destination_isocode"]
#     )

#     X = df[[
#         "Log_Origin_GDP",
#         "Log_Destination_GDP",
#         "cepii_contig",
#         "cepii_comlang_ethno",
#         "cepii_wto_o",
#         "cepii_wto_d",
#         "cepii_fta_wto",
#         "Log_cost_per_ton_seamodes",
#         "lsbci",
#         "Log_origin_production_qty",
#         "Log_destination_production_qty",
#         "country_pair_id",
#         "product_code"
#     ]]

#     ln_trade_pred = trade_ann_pipeline.predict(X)
#     df["tons_seamode_pred"] = np.exp(ln_trade_pred)

#     return df


In [ ]:
# feature_ann_cols = feature_ann_cols
# categorical_ann_cols = categorical_ann_cols

# numerical_features = numerical_features
# trade_categorical_features = trade_categorical_features

# 2. SCENARIO CONFIGURATION

In [3]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

baseline_cfg = load_json("configs/baseline_config.json")

BASE_YEAR = baseline_cfg["base_year"]
FUTURE_YEARS = baseline_cfg["future_years"]
baseline_growth = baseline_cfg["baseline_growth"]

print("Baseline loaded:")
print("Base year:", BASE_YEAR)
print("Future years:", FUTURE_YEARS)
print("Growth assumptions:", baseline_growth)

Baseline loaded:
Base year: 2024
Future years: [2025, 2026, 2027, 2028, 2029, 2030]
Growth assumptions: {'gdp_growth': 0.016, 'population_growth': 0.013, 'production_growth': 0.0, 'calorific_growth': 0.0, 'lsbci_growth': 0.0, 'bdi_growth': 0.0, 'fbx_growth': 0.0}


For stress/shock test can load pre-saved configuration/case

In [31]:
scenario_file = "configs/eu_ets.json"  # Change scenario here
scenario_config = load_json(scenario_file)
loaded_scenario_name = scenario_config["scenario_name"]
print("Loaded scenario:", scenario_config["scenario_name"])

Loaded scenario: EU ETS Maritime Regulation


## Scenario configuration

if not scenario_config.get("affected_corridors"):

    print("Computing affected corridors dynamically...")

    # Build temporary baseline
    df_future_temp = build_future_features(df_fwd_forecast_gapyear)

    df_base_temp = predict_transport_cost(
        df_future_temp,
        make_ann_features_for_forecast,
        loaded_model_cost_ann,
        feature_ann_cols,
        categorical_ann_cols
    )

    df_base_temp = predict_trade_volume(
        df_base_temp,
        numerical_features,
        trade_categorical_features,
        trade_ann_pipeline
    )

    def eligibility_rule(df):
        return df["origin_isocode"].isin(
            scenario_config.get("origin_filter", [])
        )

    affected_corridors = select_affected_corridors(
        df_baseline=df_base_temp,
        eligibility_rule=eligibility_rule,
        product_codes=scenario_config.get("affected_product_codes", []),
        share_threshold=scenario_config.get("share_threshold", 0.85),
        start_year=2025,
        end_year=2030
    )

    scenario_config["affected_corridors"] = affected_corridors

    print("Selected corridors:", len(affected_corridors))

# Run simulation

In [32]:
results = run_baseline_and_scenario(
    df_fwd_forecast_gapyear=df_fwd_forecast_gapyear,
    scenario_config=scenario_config,

    # ---- COST MODEL ----
    make_ann_features_for_forecast=ann_feature_wrapper,
    loaded_model_cost_ann=loaded_model_cost_ann,
    feature_ann_cols=COST_NUMERIC_FEATURES,
    categorical_ann_cols=COST_CATEGORICAL_FEATURES,

    # ---- TRADE MODEL ----
    numerical_features=TRADE_NUMERIC_FEATURES,
    trade_categorical_features=TRADE_CATEGORICAL_FEATURES,
    trade_ann_pipeline=trade_ann_pipeline,

    # ---- BASELINE CONFIG ----
    baseline_config=baseline_growth,
    base_year=BASE_YEAR,
    future_years=FUTURE_YEARS,

    output_dir="results"
)
# -------------------------------------------
# Merge FOB reference into compare dataset
# -------------------------------------------

df_fob_ref = (
    df_bltr_multivar
    .loc[
        df_bltr_multivar["fob_per_ton_seamodes"].notna(),
        ["origin_isocode", "destination_isocode", "product_code", "fob_per_ton_seamodes"]
    ]
    .groupby(
        ["origin_isocode", "destination_isocode", "product_code"],
        as_index=False
    )
    .agg(fob_per_ton_ref=("fob_per_ton_seamodes", "median"))
)

results["df_compare"] = (
    results["df_compare"]
    .merge(
        df_fob_ref,
        on=["origin_isocode", "destination_isocode", "product_code"],
        how="left"
    )
)
df_baseline = results["df_baseline"]
df_stress = results["df_stress"]
df_compare = results["df_compare"]

print(f"Simulation scenario {loaded_scenario_name} complete.")

Simulation scenario EU ETS Maritime Regulation complete.


Run this to save the results.

In [29]:
from pathlib import Path
import json

def save_scenario_results(results, scenario_name, output_root="results"):
    base = Path(output_root) / scenario_name
    base.mkdir(parents=True, exist_ok=True)

    # Save data
    results["df_baseline"].to_parquet(base / "baseline.parquet", index=False)
    results["df_stress"].to_parquet(base / "scenario.parquet", index=False)
    results["df_compare"].to_parquet(base / "compare.parquet", index=False)

    # Save config
    with open(base / "config.json", "w") as f:
        json.dump({
            "baseline_config": baseline_growth,
            "scenario_config": scenario_config
        }, f, indent=2)

    print(f"Saved to {base}")
from pathlib import Path
import json

def export_for_dashboard(results, scenario_slug, root="results"):
    out_dir = Path(root) / scenario_slug
    out_dir.mkdir(parents=True, exist_ok=True)

    results["df_baseline"].to_json(
        out_dir / "baseline.json",
        orient="records"
    )

    results["df_stress"].to_json(
        out_dir / "scenario.json",
        orient="records"
    )

    results["df_compare"].to_json(
        out_dir / "compare.json",
        orient="records"
    )

    with open(out_dir / "config.json", "w") as f:
        json.dump(results["config"], f, indent=2)


In [ ]:
df_compare

,origin_isocode,destination_isocode,product_code,year,transport_cost_per_ton_pred_baseline,tons_seamode_pred_baseline,transport_cost_per_ton_pred_stress,tons_seamode_pred_stress,pct_cost_change,pct_trade_change
0,AFG,PAK,1008,2025,330.025774,12.074060,330.026654,12.074040,0.000267,-0.000170
1,AFG,PAK,1005,2025,399.104857,7.987187,399.105744,7.987170,0.000222,-0.000221
2,ALB,MNE,1006,2025,23.733867,33.306884,23.733825,33.307024,-0.000180,0.000420
3,DZA,FRA,1006,2025,13.521106,1.586330,13.521380,1.586313,0.002031,-0.001045
4,DZA,FRA,1001,2025,9.155112,1.287330,9.155089,1.287332,-0.000243,0.000176
...,...,...,...,...,...,...,...,...,...,...
154939,ZWE,USA,1005,2030,15.021895,562.737019,15.021878,562.738680,-0.000113,0.000295
154940,ZWE,ZMB,1005,2030,19.734218,221.769774,19.734257,221.768012,0.000196,-0.000794
154941,AFG,KAZ,1006,2030,163.659519,10.238829,163.659765,10.238818,0.000150,-0.000104
154942,AFG,MOZ,1005,2030,549.274603,27.333176,549.273932,27.333201,-0.000122,0.000092


In [33]:
save_scenario_results(results, scenario_name=scenario_config["scenario_name"])
export_for_dashboard(results, scenario_config["scenario_name"])

Saved to results/EU ETS Maritime Regulation


In [16]:
df_fwd_forecast_gapyear.to_json(
        "gapyear.json",
        orient="records"
    )
df_fob_ref.to_json(
        "fob_reference.json",
        orient="records"
    )

,origin_isocode,destination_isocode,product_code,fob_per_ton_ref
0,AFG,ARE,1001,500.000000
1,AFG,KAZ,1006,1000.000000
2,AFG,MOZ,1005,1150.312629
3,AFG,MOZ,1006,884.173298
4,AFG,NER,1008,1970.879802
...,...,...,...,...
14843,ZWE,USA,1005,856.684982
14844,ZWE,ZMB,1001,675.057208
14845,ZWE,ZMB,1005,3394.247386
14846,ZWE,ZMB,1006,194.698355


# Scenario Analytics

In [ ]:
def load_scenario_results(scenario_name, input_root="results"):
    base = Path(input_root) / scenario_name
    return {
        "df_baseline": pd.read_parquet(base / "baseline.parquet"),
        "df_scenario": pd.read_parquet(base / "scenario.parquet"),
        "df_compare": pd.read_parquet(base / "compare.parquet"),
        "config": json.load(open(base / "config.json"))
    }

results_black_sea = load_scenario_results("black_sea")
results_el_nino   = load_scenario_results("el_nino")
results_eu_ets    = load_scenario_results("eu_ets")

df_fob_ref = (
    df_bltr_multivar
    .loc[df_bltr_multivar["fob_per_ton_seamodes"].notna()]
    .groupby(["origin_isocode","destination_isocode","product_code"], as_index=False)
    .agg(fob_per_ton_ref=("fob_per_ton_seamodes","median"))
)
def prepare_analytics_df(results_dict, df_fob_ref):
    dfs = []

    for case, res in results_dict.items():
        df = res["df_compare"].copy()
        df["case"] = case

        df = df.merge(
            df_fob_ref,
            on=["origin_isocode","destination_isocode","product_code"],
            how="left"
        )

        df["tc_fob_ratio_baseline"] = (
            df["transport_cost_per_ton_pred_baseline"] / df["fob_per_ton_ref"]
        )

        df["tc_fob_ratio_stress"] = (
            df["transport_cost_per_ton_pred_stress"] / df["fob_per_ton_ref"]
        )

        df["delta_tc_fob_pp"] = (
            df["tc_fob_ratio_stress"] - df["tc_fob_ratio_baseline"]
        ) * 100

        df["trade_index"] = (
            df["tons_seamode_pred_stress"] /
            df["tons_seamode_pred_baseline"]
        ) * 100

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

# 8. EXPORT FOR DASHBOARD

In [ ]:
Path("dashboard_data").mkdir(exist_ok=True)
df_compare.to_parquet("dashboard_data/scenario_compare.parquet")

print("Simulation completed successfully.")

In [10]:
!python -m http.server 8000

Serving HTTP on :: port 8000 (http://[::]:8000/) ...
::1 - - [19/Feb/2026 01:21:45] "GET /dashboards/simulation.html HTTP/1.1" 200 -
::1 - - [19/Feb/2026 01:21:46] "GET /results/black_sea/baseline.json HTTP/1.1" 200 -
::1 - - [19/Feb/2026 01:21:46] code 404, message File not found
::1 - - [19/Feb/2026 01:21:46] "GET /favicon.ico HTTP/1.1" 404 -
::1 - - [19/Feb/2026 01:21:48] "GET /results/black_sea/scenario.json HTTP/1.1" 200 -
::1 - - [19/Feb/2026 01:21:49] "GET /results/black_sea/compare.json HTTP/1.1" 200 -
::1 - - [19/Feb/2026 01:25:54] "GET /dashboards/simulation.html HTTP/1.1" 200 -
::1 - - [19/Feb/2026 01:25:54] "GET /results/black_sea/baseline.json HTTP/1.1" 304 -
::1 - - [19/Feb/2026 01:25:55] "GET /results/black_sea/scenario.json HTTP/1.1" 304 -
::1 - - [19/Feb/2026 01:25:56] "GET /results/black_sea/compare.json HTTP/1.1" 304 -
::1 - - [19/Feb/2026 01:25:56] "GET /results/black_sea/config.json HTTP/1.1" 200 -
::1 - - [19/Feb/2026 01:49:23] "GET /dashboards/simulation.html HTT

# Misc. Custom Case config generator

In [26]:
from generate_euets_config import generate_eu_ets_json

generate_eu_ets_json(results["df_baseline"])

✅ EU ETS config saved to: configs/eu_ets.json


In [21]:
# from scenario_generator import build_eu_ets_scenario, save_scenario_config

# scenario_config_eu_ets = build_eu_ets_scenario(results["df_baseline"])

# save_scenario_config(scenario_config_eu_ets, "eu_ets")

NameError: name 'select_affected_corridors' is not defined

In [22]:
import sys
sys.path

['/Users/bimawicaksana/miniforge3/envs/dataenv/lib/python311.zip',
 '/Users/bimawicaksana/miniforge3/envs/dataenv/lib/python3.11',
 '/Users/bimawicaksana/miniforge3/envs/dataenv/lib/python3.11/lib-dynload',
 '',
 '/Users/bimawicaksana/miniforge3/envs/dataenv/lib/python3.11/site-packages']